# Privacy & Governance Audit

This notebook conducts a structured privacy and governance audit of the NovaCred credit decisioning system. The objective is to assess whether NovaCred's data handling practices, automated decision-making processes, and AI governance posture comply with applicable EU regulation, specifically GDPR (Reg. 2016/679), the EU AI Act (Reg. 2024/1689), and the EBA Guidelines on Loan Origination and Monitoring (EBA/GL/2020/06).

This notebook represents the detailed technical analysis for the privacy and governance pillar of the governance audit. While the README consolidates overall conclusions across all governance dimensions, this document provides the structured findings, risk assessments, and recommended controls underlying those conclusions.

This audit is constrained to what can be assessed from the dataset and repository alone. Infrastructure-level controls such as access logs, encryption at rest, and consent management systems are out of scope and are flagged explicitly in the audit limitations section.

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Audit Framework](#2-audit-framework)
3. [Methodological Principles](#3-methodological-principles)
4. [Data Loading](#4-data-loading)
5. [PII Identification & Classification](#5-pii-identification--classification)
   - 5.1 [Classification and Coverage](#51-classification-and-coverage)
   - 5.2 [PII Risk Assessment](#52-pii-risk-assessment)
6. [Pseudonymisation Demo](#6-pseudonymisation-demo)
   - 6.1 [Pseudonymizing Direct Identifiers](#61-pseudonymizing-direct-identifiers)
   - 6.2 [Pseudonymization Risk Assessment](#62-pseudonymization-risk-assessment)
7. [Re-identification Risk](#7-re-identification-risk)
   - 7.1 [Testing Re-identification](#71-testing-re-identification)
   - 7.2 [Re-identification Risk Assessment](#72-re-identification-risk-assessment)
8. [GDPR Article Mapping](#8-gdpr-article-mapping)
   - 8.1 [Art. 5 - Principles relating to processing of personal data](#81-art-5---principles-relating-to-processing-of-personal-data)
   - 8.2 [Art. 6 - Lawfulness of Processing](#82-art-6---lawfulness-of-processing)
   - 8.3 [Art. 7 - Conditions for Consent](#83-art-7---conditions-for-consent)
   - 8.4 [Art. 9 - Special Category Data](#84-art-9---special-category-data)
   - 8.5 [Art. 17 - Right to Erasure](#85-art-17---right-to-erasure)
   - 8.6 [Art. 22 - Automated Decision-Making](#86-art-22---automated-decision-making)
   - 8.7 [Art. 25 - Data Protection by Design and by Default](#87-art-25---data-protection-by-design-and-by-default)
   - 8.8 [Art. 5(1)(e) - Proposed Data Retention Schedule](#88-art-51e---proposed-data-retention-schedule)
   - 8.9 [GDPR Compliance - Consolidated Risk Assessment](#89-gdpr-compliance---consolidated-risk-assessment)
9. [EU AI Act Risk Classification](#9-eu-ai-act-risk-classification)
   - 9.1 [Risk Tier Classification](#91-risk-tier-classification)
   - 9.2 [High-Risk Obligations Assessment (Art. 9-15)](#92-high-risk-obligations-assessment-art-9-15)
10. [Governance Controls & Recommendations](#10-governance-controls--recommendations)
11. [Data Remediation](#11-data-remediation)
12. [Consolidated Risk Assessment](#12-consolidated-risk-assessment)
13. [Audit Limitations](#13-audit-limitations)
14. [Conclusions](#14-conclusions)

## 1. Executive Summary

**Overall privacy risk classification: High.**

This notebook assesses NovaCred's automated credit decisioning pipeline against GDPR and the EU AI Act using evidence available in the bias-remediated dataset and repository artifacts. The audit identifies high-impact privacy and governance gaps that increase re-identification exposure, reduce auditability, and weaken decision transparency for rejected applicants.

| Finding | Evidence | Severity |
|---|---|---|
| Direct identifiers are present in the analytical dataset | `full_name`, `email`, `ssn`, `ip_address` are present with near-complete population coverage, enabling direct re-identification without quasi-identifier combinations | Critical |
| Decision transparency is weak for rejected applications | 210 of 500 applications are rejected (42.0%). 170 of 210 rejection reasons (81.0%) use `algorithm_risk_score`, which is not meaningful for contestation or human review | Critical |
| Dataset-level auditability is limited | `processing_timestamp` is missing for 438 of 500 records (87.6%), limiting traceability and post-hoc investigation | High |
| Conditional sensitive behavioral fields are present without necessity evidence | `spending_alcohol` 11 (2.2%), `spending_gambling` 7 (1.4%), `spending_adult_entertainment` 5 (1.0%) create disproportionate privacy risk unless necessity and access restrictions are documented | High |
| Conditional health-related inference is present | `loan_purpose` is populated for 50 records (10.0%). The value `medical` appears in 8 records (1.6%), creating conditional special-category exposure risk | High |
| EU AI Act high-risk governance obligations are not evidenced | The system qualifies as high-risk under Annex III, point 5(b). Key obligations such as risk management, documentation, logging, transparency, and human oversight are not evidenced in repository artifacts | High |

The highest priority controls are to enforce privacy by default at the dataset layer by separating identity data from analytical data, replace opaque rejection reasons with a controlled reason-code taxonomy and contestation workflow, and implement complete decision audit logging with retention and deletion enforcement. A DPIA process under GDPR Art. 35 is required for automated credit decisioning that produces significant effects on applicants and should be used to document necessity, proportionality, risks, and mitigations before operational deployment.

## 2. Audit Framework

This privacy and governance assessment is structured across four regulatory frameworks, including one conditional framework. Based on the project description and repository scope, NovaCred is treated as both the provider and deployer of the credit scoring AI system for the purpose of assigning responsibilities under the EU AI Act.

| Regulation | Applicability | Key obligations | Dimension | Detail |
|---|---|---|---|---|
| GDPR (Reg. 2016/679) | Confirmed - personal data of EU residents | Lawful basis (Art. 6), data minimisation (Art. 5), data subject rights (Art. 17, 22), privacy by design (Art. 25) | System under audit | Automated credit approval and interest rate assignment |
| EU AI Act (Reg. 2024/1689) | Confirmed - credit scoring is high-risk under Annex III, Section 5(b) | Risk management (Art. 9), data governance (Art. 10), transparency (Art. 13), human oversight (Art. 14), accuracy and robustness (Art. 15) | Data scope | 500 application records (post-deduplication), 34+ columns |
| EBA ML Guidelines (EBA/GL/2020/06) | Confirmed - ML used in credit decisioning | Governance framework, explainability, bias monitoring, documentation expectations | Audit reference date | 2025-12-31 |
| 4AMLD (Dir. 2015/849) | Conditional - applies if NovaCred performs regulated KYC/AML checks | Customer due diligence and record-keeping, including retention expectations (Art. 40) | Regulatory scope | Additional requirements beyond GDPR and EU AI Act if AML obligations apply |

**Regulatory Exposure**

| Regulation | Applicability | Max penalty (indicative) | Key references | Audit status (dataset and repository evidence only) |
|---|---|---|---|---|
| GDPR (Reg. 2016/679) | Confirmed | Up to EUR 20M or 4% global turnover | Art. 5, 6, 9, 17, 22, 25, 35 | Multiple high-risk gaps are indicated by dataset-level evidence |
| EU AI Act (Reg. 2024/1689) | Confirmed - high-risk | Up to EUR 15M or 3% global turnover (depending on infringement category) | Art. 9-15, Annex III Section 5(b) | Assessed in Section 9 against high-risk obligations using available evidence |
| EBA ML Guidelines (EBA/GL/2020/06) | Confirmed | Supervisory measures (non-monetary) | Governance and ML use expectations | Partially assessable without model documentation and monitoring artefacts |
| 4AMLD (Dir. 2015/849) | Conditional | National penalties | Art. 40 | Cannot be confirmed from the dataset alone |

Issues are classified using the following severity scale:

| Level | Definition |
|---|---|
| Low | Minor gap with limited regulatory exposure; addressable through routine process improvement |
| Moderate | Relevant compliance gap requiring correction but unlikely to trigger enforcement action in isolation |
| High | Significant gap likely to draw regulatory scrutiny or materially weaken data subject protections |
| Critical | Severe gap with material enforcement risk, requiring immediate remediation before next deployment |

## 3. Methodological Principles

| Principle | Description |
|---|---|
| **Reproducibility** | Fixed random seed (42); all transforms deterministic and documented |
| **Evidence-based** | Every finding supported by computed metrics or explicit regulatory text |
| **Regulatory mapping** | All findings mapped to specific GDPR articles or EU AI Act provisions |
| **Proportionality** | Severity ratings aligned to regulatory risk classification standards |
| **Pipeline integrity** | Notebook loads from `bias_remediated_credit_applications.parquet` (output of `02-bias-analysis.ipynb`); protected attributes and proxy variables are already absent |


## 4. Data Loading

This notebook loads the bias-remediated dataset produced by `02-bias-analysis.ipynb`. Protected attributes (`gender`, `date_of_birth`, `age`) and key proxy variables (`zip_code`) were present in the raw dataset but were removed upstream as part of the fairness remediation. For completeness, these removed fields are still referenced in the PII classification discussion to meet the audit requirement of identifying all relevant PII categories, even when not available for further analysis in this notebook.

The focus of this notebook is the residual privacy and governance risk in the remaining data, including direct identifiers (for example `full_name`, `email`, `ssn`, `ip_address`), conditional sensitive signals (for example `loan_purpose` values such as `medical`), and evidence of governance controls such as transparency, retention, and decision explainability.

In [1]:
# importing necessary libraries and configuring the environment for data analysis and visualization.
import hashlib
import logging
import os
import re
import sys
import hmac

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# add project root to path so src/ is importable from notebooks/
sys.path.insert(0, os.path.abspath(".."))

# configure logging to surface info-level messages from the data loader
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
)

# reproducibility seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# figure output directory, created if it does not already exist
FIGURES_DIR = "../reports/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


In [2]:
# Load the bias-remediated dataset produced by 02-bias-analysis.ipynb.
# Protected attributes (gender, gender_clean, date_of_birth, dob_parsed, age, age_group) and proxy variables (zip_code) are already removed

df = pd.read_parquet("../data/processed/bias_remediated_credit_applications.parquet")

print(f"Loaded bias-remediated dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns present ({df.shape[1]}):")
for col in df.columns:
    print(f"  {col}")

Loaded bias-remediated dataset: 500 rows × 38 columns

Columns present (38):
  id
  processing_timestamp
  loan_purpose
  notes
  full_name
  email
  ssn
  ip_address
  annual_income
  credit_history_months
  debt_to_income
  savings_balance
  spending_shopping
  spending_rent
  spending_alcohol
  loan_approved
  rejection_reason
  spending_dining
  spending_healthcare
  interest_rate
  approved_amount
  spending_fitness
  spending_entertainment
  spending_insurance
  spending_travel
  spending_transportation
  spending_utilities
  spending_groceries
  spending_education
  spending_adult_entertainment
  spending_gambling
  annual_salary
  date_of_birth_parsed
  total_monthly_spending
  total_annual_spending
  annual_income_clean
  annual_income_num
  region


## 5. PII Identification & Classification

Under GDPR Art. 4(1), personal data is any information relating to an identified or identifiable natural person, including identification that is possible indirectly when attributes are combined. This section classifies the columns in the bias-remediated dataset into four tiers to assess re-identification risk and regulatory sensitivity.

The classification uses the following tiers. Direct identifiers can uniquely identify an applicant on their own. Quasi-identifiers are not unique individually but increase identifiability when combined with other attributes. Sensitive or conditional sensitive fields are attributes that can enable inference about protected or special-category characteristics depending on context and therefore require heightened governance scrutiny. Non-PII refers to fields that do not materially increase identifiability within this dataset, based on the available columns.

Three quasi-identifiers (`date_of_birth`, `zip_code`, `gender`) were present in the raw dataset and flagged during the bias audit (`02-bias-analysis.ipynb`). They were removed upstream during bias remediation, but they are still included in the classification table for completeness and traceability. The classification is followed by coverage checks that quantify the exposure from the fields that remain in the dataset used in this notebook.

### 5.1 Classification and Coverage

The classification table below assigns each privacy-relevant field to a tier with its regulatory basis. It is followed by population coverage checks that quantify the re-identification surface area for the fields still present in the bias-remediated dataset.

| Field | Tier | GDPR Basis | Notes |
|---|---|---|---|
| `full_name` | Direct identifier | Art. 4(1) | Uniquely identifies an applicant; should be removed or pseudonymised before any sharing |
| `email` | Direct identifier | Art. 4(1) | Unique contact address; personal data under GDPR |
| `ssn` | Direct identifier | Art. 4(1); national identifier | Highest re-identification risk; requires strict access control and strong protection |
| `ip_address` | Direct identifier (online identifier) | Art. 4(1) | Online identifier; personal data in context because it can be linkable to an applicant by the controller |
| `date_of_birth` | Quasi-identifier | Art. 4(1) | In combination with other attributes, enables re-identification (Sweeney 2000). Removed upstream by `02-bias-analysis.ipynb` |
| `zip_code` | Quasi-identifier | Art. 4(1) | Geographic quasi-identifier and common proxy for socio-demographic attributes in lending contexts. Removed upstream by `02-bias-analysis.ipynb` |
| `gender` | Protected attribute (personal data) | Art. 4(1) | Personal data and protected characteristic in equality law; removed upstream by `02-bias-analysis.ipynb` |
| `spending_adult_entertainment` | Conditional sensitive (behavioral) | Art. 5(1)(c) | May enable inference of sensitive characteristics depending on context; inclusion requires necessity and proportionality justification |
| `spending_gambling` | Conditional sensitive (behavioral) | Art. 5(1)(c) | Potentially sensitive behavioral signal; inclusion requires necessity and proportionality justification |
| `spending_alcohol` | Conditional sensitive (behavioral) | Art. 5(1)(c) | Potentially sensitive behavioral signal; inclusion requires necessity and proportionality justification |
| `loan_purpose` | Conditional sensitive | Art. 4(1); Art. 9(1) conditional | When `loan_purpose = "medical"`, it can reveal health-related information. Present for 50 records (10.0%); 8 records contain `medical` |
| `annual_income`, `credit_history_months`, `debt_to_income`, `savings_balance` | Financial attributes (personal data) | Art. 4(1) | Personal data used for underwriting. In credit origination, processing is commonly justified under Art. 6(1)(b) (steps prior to entering a contract) or Art. 6(1)(f) (legitimate interests), subject to the controller’s documented lawful basis and necessity assessment |
| `loan_approved`, `interest_rate`, `approved_amount`, `rejection_reason` | Decision outputs (personal data) | Art. 4(1); Art. 22 | Automated decision outputs are personal data and fall under automated decision-making safeguards |
| `notes` | Operational metadata | Art. 4(1) | Present for 2 records only; contains processing flags (RESUBMISSION, DUPLICATE_ENTRY_ERROR) |
| `processing_timestamp` | Operational metadata | Art. 5(2) | Accountability support for audit trails; 87.6% missing, limiting traceability and post-hoc auditability |

In [3]:
# check population coverage for each direct identifier high coverage means re-identification risk applies to nearly all records

direct_id_cols = ["full_name", "email", "ssn", "ip_address"]

coverage = {
    col: {
        "present": df[col].notna().sum(),
        "missing": df[col].isna().sum(),
        "coverage_pct": round(df[col].notna().mean() * 100, 1),
    }
    for col in direct_id_cols
}

coverage_df = pd.DataFrame(coverage).T
print("Direct identifier coverage (n=500 after deduplication):")
display(coverage_df)

Direct identifier coverage (n=500 after deduplication):


,present,missing,coverage_pct
full_name,500.0,0.0,100.0
email,492.0,8.0,98.4
ssn,496.0,4.0,99.2
ip_address,496.0,4.0,99.2


**Findings**

All four direct identifiers are present at near-complete coverage. `full_name` and `email` are populated for over 98% of records, `ip_address` for 99.6%, and `ssn` for 99.0%. This means that any single field is sufficient to re-identify the vast majority of applicants without requiring combinations with other attributes. The absence of pseudonymisation at ingestion indicates potential non-compliance with GDPR Art. 25 (data protection by design) and increases exposure under GDPR Art. 5(1)(f) if the dataset is accessed or shared without controls.

In [4]:
# check population coverage for sensitive spending categories
# even sparse presence requires a documented lawful basis under GDPR
sensitive_cols = ["spending_adult_entertainment", "spending_gambling", "spending_alcohol"]

sensitive_coverage = {
    col: {
        "records_with_data": df[col].notna().sum(),
        "coverage_pct": round(df[col].notna().mean() * 100, 1),
        "mean_amount": round(df[col].mean(), 2),
    }
    for col in sensitive_cols
    if col in df.columns
}

print("Sensitive spending category coverage:")
display(pd.DataFrame(sensitive_coverage).T)

Sensitive spending category coverage:


,records_with_data,coverage_pct,mean_amount
spending_adult_entertainment,5.0,1.0,591.20
spending_gambling,7.0,1.4,457.14
spending_alcohol,11.0,2.2,477.09


**Findings**

The three sensitive spending categories are sparsely populated, each appearing in fewer than 3% of records. `spending_alcohol` appears in 11 records (2.2%), `spending_gambling` in 7 records (1.4%), and `spending_adult_entertainment` in 5 records (1.0%). Even with low coverage, these attributes can enable inference about sensitive characteristics depending on context and therefore carry disproportionate privacy risk relative to likely underwriting value. Their presence should be justified under the data minimisation principle in GDPR Art. 5(1)(c) and controlled through strict access and purpose limitation.

In [5]:
# systematic scan of categorical columns to identify potentially sensitive values
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

value_preview = []
for col in categorical_cols:
    top_vals = (
        df[col]
        .dropna()
        .astype(str)
        .str.lower()
        .value_counts()
        .head(8)
        .to_dict()
    )
    value_preview.append(
        {
            "field": col,
            "non_null_records": int(df[col].notna().sum()),
            "top_values": top_vals,
        }
    )

preview_df = pd.DataFrame(value_preview).sort_values("non_null_records", ascending=False)
preview_df.head(12)

,field,non_null_records,top_values
0,id,500,"{'app_200': 1, 'app_339': 1, 'app_112': 1, 'ap..."
3,full_name,500,"{'susan flores': 3, 'george clark': 2, 'janet ..."
9,region,499,"{'nyc (10xxx)': 251, 'la (90xxx)': 230, 'other..."
5,ssn,496,"{'937-72-8731': 2, '780-24-9300': 2, '596-64-4..."
6,ip_address,496,"{'192.168.48.155': 1, '192.168.183.52': 1, '10..."
7,annual_income,495,"{'79000': 11, '100000': 10, '98000': 10, '7500..."
4,email,492,"{'jerry.smith17@hotmail.com': 1, 'patricia.lop..."
8,rejection_reason,208,"{'algorithm_risk_score': 169, 'insufficient_cr..."
1,loan_purpose,50,"{'medical': 8, 'education': 7, 'vacation': 6, ..."
2,notes,1,{'resubmission': 1}


**Findings**

A systematic scan of categorical fields was used to identify values that may indicate conditional sensitive information. This informed the focused checks below, including the value distribution of `loan_purpose`.

In [6]:
# loan purpose coverage and medical count
n_records = len(df)

lp = df["loan_purpose"].dropna().astype(str).str.lower()
total_lp = int(lp.shape[0])
medical_lp = int((lp == "medical").sum())

print(f"loan_purpose populated: {total_lp}/{n_records} ({total_lp/n_records:.1%})")
print(f'loan_purpose = "medical": {medical_lp}/{n_records} ({medical_lp/n_records:.1%})')

loan_purpose populated: 50/500 (10.0%)
loan_purpose = "medical": 8/500 (1.6%)


**Findings**

`loan_purpose` is populated for 50 records (10.0% of the dataset). Within those, 8 records (1.6% of the dataset) contain the value `medical`, which can reveal health-related information. This creates conditional special-category inference risk and requires a documented necessity and lawful exception under GDPR Art. 9 when processed for credit underwriting.

### 5.2 PII Risk Assessment

This subsection consolidates the privacy risk implications of the identified fields and assigns severity based on re-identification exposure, regulatory sensitivity, and the degree to which the dataset demonstrates protective controls.

| Risk area | Evidence (records, percent) | Regulatory mapping | Severity | Recommended control |
|---|---|---|---|---|
| Direct identifiers present with near-complete coverage | `full_name` and `email` over 98% coverage; `ip_address` 498 records (99.6%); `ssn` 495 records (99.0%) | Art. 25 (privacy by design), Art. 5(1)(f) (confidentiality) | Critical | Remove or pseudonymise direct identifiers by default; restrict access; encrypt at rest and in transit; role-based access control |
| Conditional sensitive behavioral spending fields present | alcohol 11 (2.2%), gambling 7 (1.4%), adult entertainment 5 (1.0%) | Art. 5(1)(c) (data minimisation), purpose limitation expectations | High | Justify necessity; restrict feature access; consider removal if low underwriting value; document purpose and retention |
| Health-related inference via `loan_purpose` | `loan_purpose` populated 50 (10.0%); `medical` 8 (1.6%) | Art. 9(1) conditional; Art. 6 lawful basis plus Art. 9 exception when applicable | High | Treat `medical` as restricted; implement purpose limitation and access control; document exception basis and DPIA linkage |
| Weak audit trail coverage via `processing_timestamp` | missing 438 records (87.6%) | Art. 5(2) (accountability), auditability expectations | High | Enforce mandatory logging fields; implement immutable audit trail; align with AI Act logging expectations in later sections |
| Operational exception flags in `notes` | 2 records (0.4%) | Art. 5(1)(d) (accuracy), governance traceability | Moderate | Standardise error codes; avoid free-text; track remediation and resolution workflow |

**Overall PII risk: Critical.**

## 6. Pseudonymisation Demo

GDPR Recital 26 and Art. 25 (data protection by design) require pseudonymisation as a technical safeguard. This section demonstrates three techniques applied to direct identifiers still present in the bias-remediated dataset.

1. SHA-256 hashing applied to `ssn`. Deterministic and irreversible without the original value.
2. Keyed HMAC hashing applied to `email`. The secret key is held separately by the controller, so the output is a pseudonym rather than anonymisation.
3. IP generalisation by zeroing the host octet of IPv4 addresses to reduce precision.

Note: Hashing alone does not constitute anonymisation under GDPR. Anonymisation requires that singling out and re-identification are no longer reasonably likely given the means available.

### 6.1 Pseudomyzing Direct Identifiers

In [7]:
def pseudonymise_sha256(value: str) -> str:
    """
    Replace a string value with its SHA-256 hex digest.

    Notes
    -----
    sha-256 is deterministic, enabling internal linkage while reducing disclosure risk.
    this is pseudonymisation, not anonymisation.
    """
    if pd.isna(value):
        return pd.NA
    # encode to bytes then hash, strip whitespace to avoid digest mismatches
    return hashlib.sha256(str(value).strip().encode("utf-8")).hexdigest()


def pseudonymise_hmac(value: str, secret_key: bytes) -> str:
    """
    Replace a string value with a keyed HMAC-SHA-256 pseudonym.

    Notes
    -----
    hmac strengthens hashing for low-entropy identifiers by requiring knowledge of the secret key
    to validate candidate inputs.
    """
    if pd.isna(value):
        return pd.NA
    # compute hmac with sha-256 using the provided secret key
    msg = str(value).strip().lower().encode("utf-8")
    return hmac.new(secret_key, msg=msg, digestmod=hashlib.sha256).hexdigest()


def generalise_ip(ip: str) -> str:
    """
    Generalise an IPv4 address to its /24 subnet by zeroing the host octet.

    Notes
    -----
    this performs format-level generalisation and does not validate 0-255 ranges.
    """
    if pd.isna(ip):
        return pd.NA
    match = re.fullmatch(r"(\d{1,3}\.\d{1,3}\.\d{1,3})\.(\d{1,3})", str(ip).strip())
    if match:
        return match.group(1) + ".0"
    return pd.NA


print("pseudonymisation functions defined")

# work on a copy so the original columns are preserved for comparison
df_pseudonymised = df.copy()

# ssn: sha-256 hash
df_pseudonymised["ssn_pseudo"] = df_pseudonymised["ssn"].apply(pseudonymise_sha256)

# email: hmac-sha-256 hash
# in production, the secret key must be stored in a key management service and never hardcoded
SECRET_KEY = b"novacred-demo-key-2024"  # demonstration value only
df_pseudonymised["email_pseudo"] = df_pseudonymised["email"].apply(
    pseudonymise_hmac, secret_key=SECRET_KEY
)

# full_name: replace with application id as an opaque reference token
df_pseudonymised["full_name_pseudo"] = df_pseudonymised["id"]

# ip address: generalise to /24 subnet
df_pseudonymised["ip_generalised"] = df_pseudonymised["ip_address"].apply(generalise_ip)

print("pseudonymisation applied to ssn, email, full_name, and ip_address")

pseudonymisation functions defined
pseudonymisation applied to ssn, email, full_name, and ip_address


In [8]:
# side-by-side comparison of original and pseudonymised values
comparison_cols = [
    "id",
    "full_name",
    "full_name_pseudo",
    "ssn",
    "ssn_pseudo",
    "email",
    "email_pseudo",
]

sample = (
    df_pseudonymised[df_pseudonymised["ssn"].notna()][comparison_cols]
    .head(5)
    .reset_index(drop=True)
)

# truncate long hashes for display readability
for col in ["ssn_pseudo", "email_pseudo"]:
    sample[col] = sample[col].apply(lambda x: str(x)[:16] + "..." if pd.notna(x) else pd.NA)

print("before and after pseudonymisation (first 5 records with non-null ssn)")
display(sample)

ip_sample = (
    df_pseudonymised[df_pseudonymised["ip_address"].notna()][["ip_address", "ip_generalised"]]
    .head(5)
    .reset_index(drop=True)
)

print("ip address generalisation (host octet zeroed)")
display(ip_sample)

before and after pseudonymisation (first 5 records with non-null ssn)


,id,full_name,full_name_pseudo,ssn,ssn_pseudo,email,email_pseudo
0,app_200,Jerry Smith,app_200,596-64-4340,2caf30528c21a10e...,jerry.smith17@hotmail.com,f306c33f9028ad9d...
1,app_339,Patricia Lopez,app_339,187-21-4050,33eac4d646c459d5...,patricia.lopez90@icloud.com,7c889b5d8bc52121...
2,app_112,Cynthia Rodriguez,app_112,323-67-4611,9a269bba04b588ed...,cynthia.rodriguez5@icloud.com,bfc7c088338d7e2f...
3,app_267,Donald Adams,app_267,177-32-2203,566ea0a57175e1bd...,donald.adams15@mail.com,501c4172d969c749...
4,app_185,George Clark,app_185,804-59-4925,a37ab7bcb0aa30e2...,george.clark73@aol.com,ffc9514337144ac0...


ip address generalisation (host octet zeroed)


,ip_address,ip_generalised
0,192.168.48.155,192.168.48.0
1,192.168.183.52,192.168.183.0
2,10.134.193.125,10.134.193.0
3,10.37.119.151,10.37.119.0
4,10.178.229.242,10.178.229.0


**Findings**

The pseudonymisation outputs preserve record linkage while reducing direct disclosure. `full_name` is removed and replaced with an opaque reference token (`id`). `ssn` and `email` are transformed into fixed-length digests that cannot be reversed without the original values. IP addresses are generalised to a /24 subnet to reduce precision, but they remain linkable in an internal context and should not be retained longer than necessary.

In [9]:
# determinism checks for sha-256 pseudonymisation
test_ssn = "123-45-6789"
hash_a = pseudonymise_sha256(test_ssn)
hash_b = pseudonymise_sha256(test_ssn)

print(f"determinism check (same input, same output): {hash_a == hash_b}")
print(f"hash of '{test_ssn}': {hash_a}")

hash_c = pseudonymise_sha256("123-45-6780")
print(f"different inputs produce different outputs: {hash_a != hash_c}")

determinism check (same input, same output): True
hash of '123-45-6789': 01a54629efb952287e554eb23ef69c52097a75aecc0e3a93ca0855ab6d7a31a0
different inputs produce different outputs: True


**Findings**

SHA-256 hashing is deterministic, which supports internal consistency checks and joins across tables, but it is not anonymisation. For low-entropy identifiers such as SSNs, keyed approaches such as HMAC reduce the risk of guess-and-check attacks because an adversary cannot validate candidate inputs without access to the secret key.

### 6.2 Pseudonymization Risk Assessment

| Finding | Evidence | Regulatory reference | Severity | Governance impact | Recommended control |
|---|---|---|---|---|---|
| Direct identifiers not pseudonymised at ingestion | SSN, email, full_name, ip_address stored in plaintext in the dataset under audit | GDPR Art. 5(1)(f); Art. 25; Recital 26 | Critical | Any unauthorized access exposes fully identifiable records | Apply pseudonymisation in the ingestion pipeline by default, with role-based access to raw identifiers only when required |
| Pseudonymisation key hardcoded in source code | `SECRET_KEY = b"novacred-demo-key-2024"` is defined inline for demonstration | GDPR Art. 32; Art. 25 | High | Key exposure nullifies the protection of HMAC pseudonyms | Store keys in a dedicated key management service, rotate regularly, and separate key access from dataset access |
| IP generalisation reduces but does not eliminate re-identification | Generalised /24 values may still be linkable in internal environments | GDPR Art. 4(1) | Moderate | Residual identifiability may remain through network logs or internal device mapping | Minimise retention of raw IPs, apply stronger generalisation where possible, and restrict access to fraud and security functions only |

**Overall pseudonymisation risk: High.**

## 7. Re-identification Risk

Even after pseudonymising direct identifiers, datasets can remain re-identifiable through combinations of quasi-identifiers. Prior work by Sweeney (2000) showed that a large share of the US population can be uniquely identified using the combination of 5-digit ZIP code, date of birth, and gender.

In the NovaCred pipeline, these three quasi-identifiers were present in the raw dataset but were removed upstream during bias remediation. This section therefore evaluates re-identification risk in two steps. First, it documents the classic ZIP code plus date of birth plus gender risk as an upstream exposure. Second, it assesses the residual risk in the bias-remediated dataset, where direct identifiers remain and are sufficient to uniquely identify individuals without relying on quasi-identifier combinations.

A dataset satisfies k-anonymity if every record is indistinguishable from at least k-1 other records with respect to a chosen quasi-identifier set. k equals 1 indicates uniqueness for that quasi-identifier set and a high re-identification risk.

### 7.1 Testing Re-identification

In [10]:
# k-anonymity style uniqueness checks on identifier and attribute combinations
# note: zip_code, date_of_birth, and gender were removed upstream in 02-bias-analysis.ipynb

def k_anonymity_summary(df_in: pd.DataFrame, cols: list[str]) -> dict:
    existing = [c for c in cols if c in df_in.columns]
    if len(existing) != len(cols):
        return {
            "attribute_set": " + ".join(cols),
            "records_complete": 0,
            "distinct_groups": 0,
            "k_min": pd.NA,
            "unique_groups": 0,
            "unique_group_percent": pd.NA,
            "status": "skipped (missing columns)",
        }

    d = df_in[cols].dropna()
    if len(d) == 0:
        return {
            "attribute_set": " + ".join(cols),
            "records_complete": 0,
            "distinct_groups": 0,
            "k_min": pd.NA,
            "unique_groups": 0,
            "unique_group_percent": pd.NA,
            "status": "no complete records",
        }

    group_sizes = d.groupby(cols).size()
    k_min = int(group_sizes.min())
    unique_groups = int((group_sizes == 1).sum())
    distinct_groups = int(len(group_sizes))
    unique_pct = round(unique_groups / distinct_groups * 100, 1) if distinct_groups > 0 else pd.NA

    return {
        "attribute_set": " + ".join(cols),
        "records_complete": int(len(d)),
        "distinct_groups": distinct_groups,
        "k_min": k_min,
        "unique_groups": unique_groups,
        "unique_group_percent": f"{unique_pct:.1f}%" if unique_pct is not pd.NA else pd.NA,
        "status": "ok",
    }

attribute_sets = [
    ["full_name"],
    ["email"],
    ["ssn"],
    ["ip_address"],
    ["ip_address", "loan_purpose"],
    ["loan_purpose", "loan_approved"],
]

rows = [k_anonymity_summary(df, cols) for cols in attribute_sets]
reident_df = pd.DataFrame(rows)

display(reident_df)

,attribute_set,records_complete,distinct_groups,k_min,unique_groups,unique_group_percent,status
0,full_name,500,475,1,451,94.9%,ok
1,email,492,492,1,492,100.0%,ok
2,ssn,496,494,1,492,99.6%,ok
3,ip_address,496,496,1,496,100.0%,ok
4,ip_address + loan_purpose,50,50,1,50,100.0%,ok
5,loan_purpose + loan_approved,50,19,1,2,10.5%,ok


**Findings**

The upstream quasi-identifier triple ZIP code plus date of birth plus gender is not testable in this notebook because those columns were removed during bias remediation. In the bias-remediated dataset, the residual re-identification risk is driven by direct identifiers. The k-anonymity summaries show that single-field identifier sets such as `full_name`, `email`, and `ssn` produce k equal to 1, which means individual records are unique without requiring any attribute combination. As a result, any sharing or broad access to this dataset without strict protection would expose identifiable applicant records.

**Method note on k-anonymity limitations**

k-anonymity is a useful first diagnostic, but it is not sufficient to guarantee privacy. Even when k is greater than 1, datasets can remain vulnerable to homogeneity attacks where sensitive attributes are the same within a group, and background knowledge attacks where an adversary combines external information with quasi-identifiers. Stronger protections include l-diversity, which requires diversity of sensitive values within each group, and t-closeness, which limits how much the sensitive value distribution within a group can deviate from the overall distribution. These extensions are relevant if NovaCred intends to share analytical datasets externally.

### 7.2 Re-identification Risk Assessment

| Finding | Evidence | Regulatory reference | Severity | Governance impact | Recommended control |
|---|---|---|---|---|---|
| Direct identifiers yield k equal to 1 | Single-field sets such as `full_name`, `email`, and `ssn` form unique groups, indicating immediate identifiability | GDPR Art. 4(1); Art. 25; Recital 26 | Critical | Re-identification does not require quasi-identifiers, so accidental exposure would directly identify applicants | Remove or pseudonymise direct identifiers by default; restrict access to raw identifiers; log access and enforce least privilege |
| Attribute combinations can amplify uniqueness when identifiers remain | Combinations including `ip_address` and `loan_purpose` can create additional unique groups among complete records | GDPR Art. 5(1)(c) | High | External sharing of even partially de-identified extracts remains risky when identifiers are present | Apply data minimisation and suppression before sharing; enforce k thresholds for extracts; use aggregate reporting where possible |
| k-anonymity alone does not guarantee privacy | k-anonymity can still allow homogeneity and background knowledge attacks | GDPR Recital 26 | Moderate | Over-reliance on k may lead to false confidence in anonymisation | Use l-diversity or t-closeness where sensitive inference is plausible; consider differential privacy for published statistics |

**Overall re-identification risk: Critical.**

## 8. GDPR Article Mapping

This section maps the dataset-level findings to specific GDPR obligations that are relevant for automated credit decisioning. The assessment is constrained to what can be evidenced from the dataset and repository artifacts. Where controller-side documentation is required (lawful basis records, consent logs, retention policy, and data subject request workflows), missing evidence is treated as a governance gap.

### 8.1 Art. 5 - Principles relating to processing of personal data

Art. 5 defines core processing principles, including purpose limitation, data minimisation, accuracy, storage limitation, and integrity and confidentiality. In this notebook, the Art. 5 assessment focuses on whether the dataset contains fields that are unnecessary for credit underwriting or disproportionately risky relative to their likely utility.

In [11]:
# evidence for data minimisation concerns (direct identifiers and conditional sensitive fields)
n_records = len(df)

direct_id_cols = ["full_name", "email", "ssn", "ip_address"]
direct_non_null = df[direct_id_cols].notna().sum()
direct_pct = (direct_non_null / n_records * 100).round(1)

sensitive_cols = ["spending_adult_entertainment", "spending_gambling", "spending_alcohol"]
sensitive_non_null = df[sensitive_cols].notna().sum()
sensitive_pct = (sensitive_non_null / n_records * 100).round(1)

min_df = pd.DataFrame(
    {
        "non_null_records": pd.concat([direct_non_null, sensitive_non_null]),
        "coverage_percent": pd.concat([direct_pct, sensitive_pct]),
    }
)

display(min_df)

,non_null_records,coverage_percent
full_name,500,100.0
email,492,98.4
ssn,496,99.2
ip_address,496,99.2
spending_adult_entertainment,5,1.0
spending_gambling,7,1.4
spending_alcohol,11,2.2


**Findings**

Direct identifiers are present at near-complete coverage, which materially increases exposure under the integrity and confidentiality principle in Art. 5(1)(f). Conditional sensitive spending fields are sparsely populated but create disproportionate privacy risk relative to likely underwriting value, which indicates potential non-compliance with the data minimization principle in Art. 5(1)(c) unless necessity is documented and access is tightly restricted.

### 8.2 Art. 6 - Lawfulness of Processing

Art. 6 requires that personal data processing is linked to a documented lawful basis. Credit underwriting commonly relies on contract necessity (steps prior to entering a contract) or legitimate interests, depending on the controller’s documented rationale and necessity assessment. This notebook cannot verify the lawful basis from the dataset alone, so evidence is assessed as the presence or absence of fields that support accountability and traceability.

| Field Group | Legal Basis | Compliant | Rationale |
|---|---|---|---|
| Financial variables (income, DTI, savings, credit history) | Art. 6(1)(b) — contract performance | **YES** | Core underwriting variables directly necessary to assess creditworthiness |
| `full_name`, `email` | Art. 6(1)(b) — contract performance | **CONDITIONAL** | Required for identification and communication; retention beyond decision must be justified |
| `ssn` | Art. 6(1)(c) — legal obligation (AML/KYC) | **CONDITIONAL** | Potentially justified under AML compliance but must be documented; 5 records missing SSN |
| `ip_address` | Art. 6(1)(f) — legitimate interests | **UNCERTAIN** | Fraud prevention is a legitimate interest but requires a documented balancing test |
| Sensitive spending (`gambling`, `adult_entertainment`, `alcohol`) | No documented basis | **NO** | No clear credit relevance; collecting these features without a lawful basis violates Art. 6 |


In [12]:
# check for evidence of lawful basis documentation fields in the dataset
# examples: lawful_basis, consent_status, consent_timestamp, privacy_notice_version
candidate_cols = [
    "lawful_basis",
    "consent_status",
    "consent_timestamp",
    "privacy_notice_version",
    "privacy_notice_timestamp",
    "terms_accepted",
    "terms_accepted_timestamp",
]

present = [c for c in candidate_cols if c in df.columns]
missing = [c for c in candidate_cols if c not in df.columns]

print(f"candidate accountability fields present: {present}")
print(f"candidate accountability fields missing: {missing}")

candidate accountability fields present: []
candidate accountability fields missing: ['lawful_basis', 'consent_status', 'consent_timestamp', 'privacy_notice_version', 'privacy_notice_timestamp', 'terms_accepted', 'terms_accepted_timestamp']


**Findings**

No dataset fields indicate the lawful basis used for processing or provide traceability to the applicable privacy notice or contract step. This does not prove unlawful processing, but it indicates a governance gap because the controller’s Art. 6 justification cannot be evidenced or audited from repository artifacts.

### 8.3 Art. 7 — Conditions for Consent

Where NovaCred cannot rely on Art. 6(1)(b) (contract performance) or Art. 6(1)(c) (legal obligation), explicit consent under Art. 6(1)(a) / Art. 9(2)(a) is the only remaining lawful basis, for example, for sensitive spending categories and any processing that extends beyond core underwriting.

GDPR Art. 7 imposes five cumulative requirements for valid consent:

| Requirement | Art. 7 Reference | NovaCred Status | Gap |
|---|---|---|---|
| **Freely given** — consent must not be a condition of service | Art. 7(4) | NOT EVIDENCED | Bundling consent with credit application would invalidate it |
| **Specific** — separate consent per distinct processing purpose | Art. 7(2) | NOT EVIDENCED | A single omnibus consent covering all data types is invalid |
| **Informed** — Art. 13/14 transparency notice provided at collection | Art. 7(1); Art. 13 | NOT EVIDENCED | No privacy notice or data subject information sheet found in repository |
| **Unambiguous** — affirmative opt-in act required | Art. 4(11) | NOT EVIDENCED | Pre-ticked boxes or continued use do not constitute consent |
| **Withdrawable** — withdrawal must be as easy as giving consent | Art. 7(3) | NOT EVIDENCED | No withdrawal mechanism or erasure trigger documented |

**Consent for Automated Decision-Making (Art. 22(2)(c)):** Where NovaCred relies on explicit consent as the basis for automated credit decisions, that consent must be collected separately from any other consent, documented with a timestamp and notice version, and linked to a functioning withdrawal endpoint that triggers the Art. 17 erasure workflow.

**Verdict:** No valid consent mechanism is evidenced. NovaCred cannot rely on consent as a fallback lawful basis for sensitive spending data or automated decisions without first implementing a compliant consent framework. Processing of these fields remains without a documented lawful basis until that framework is in place.

In [13]:
# evidence for consent tracking
consent_cols = [c for c in df.columns if "consent" in c.lower()]
display(pd.DataFrame({"consent_related_columns": consent_cols}))

,consent_related_columns


**Findings**

No consent tracking fields are present in the dataset. If NovaCred relies on consent for any data sources or secondary purposes, consent cannot be audited from repository artifacts. If consent is not used, the governance requirement shifts to documenting the actual lawful basis and ensuring purpose limitation for all collected attributes.

### 8.4 Art. 9 - Special Category Data

Art. 9 restricts processing of special category data, including health data. Some attributes are not explicitly special category data but can enable inference of health-related information depending on values and context. This subsection assesses conditional Art. 9 exposure in the bias-remediated dataset used in this notebook.

Fields such as `gender`, `date_of_birth`, `age`, and `zip_code` were present in the raw dataset but were removed upstream during bias remediation. As a result, this section focuses on residual conditional exposure that remains observable in the current dataset.

In [14]:
# conditional Art. 9 exposure via loan_purpose values
n_records = len(df)

lp = df["loan_purpose"].dropna().astype(str).str.lower() if "loan_purpose" in df.columns else pd.Series([], dtype="object")
total_lp = int(lp.shape[0])
medical_lp = int((lp == "medical").sum()) if total_lp > 0 else 0

print(f"loan_purpose populated: {total_lp}/{n_records} ({total_lp/n_records:.1%})")
print(f'loan_purpose = "medical": {medical_lp}/{n_records} ({medical_lp/n_records:.1%})')

loan_purpose populated: 50/500 (10.0%)
loan_purpose = "medical": 8/500 (1.6%)


| Field | Conditional special-category exposure | Evidence (records, percent) | Art. 9 exception evidence | Severity |
|---|---|---|---|---|
| `loan_purpose` | Health-related inference when value is `medical` | `loan_purpose` populated 50 (10.0%); `medical` 8 (1.6%) | Not evidenced in dataset or repository | High |
| `spending_alcohol`, `spending_gambling`, `spending_adult_entertainment` | Potential inference of sensitive lifestyle or health-related attributes depending on context | alcohol 11 (2.2%), gambling 7 (1.4%), adult entertainment 5 (1.0%) | Not evidenced in dataset or repository | High |

**Verdict:** Conditional Art. 9 exposure is present through `loan_purpose = medical` and potentially through sensitive spending signals. No evidence of an applicable Art. 9 exception or necessity assessment is available in repository artifacts, indicating a high governance risk.

### 8.5 Art. 17 - Right to Erasure

Art. 17 requires the controller to delete personal data in applicable cases and to operationalise deletion across storage locations, derived datasets, and downstream consumers. From a dataset perspective, erasure feasibility depends on the ability to locate records, identify linked artifacts, and verify when processing occurred. This subsection evaluates whether the dataset contains the minimum structure needed to support traceability for deletion requests.

In [15]:
# evidence for erasure feasibility and traceability
n_records = len(df)

trace_cols = ["id", "processing_timestamp"]
present_trace_cols = [c for c in trace_cols if c in df.columns]

print(f"dataset size: {n_records}")
print(f"traceability columns present: {present_trace_cols}")

if "id" in df.columns:
    id_missing = int(df["id"].isna().sum())
    id_unique = int(df["id"].nunique(dropna=True))
    print(f"id missing: {id_missing}/{n_records} ({id_missing/n_records:.1%})")
    print(f"id unique: {id_unique}/{n_records} ({id_unique/n_records:.1%})")

if "processing_timestamp" in df.columns:
    ts_missing = int(df["processing_timestamp"].isna().sum())
    print(f"processing_timestamp missing: {ts_missing}/{n_records} ({ts_missing/n_records:.1%})")

dataset size: 500
traceability columns present: ['id', 'processing_timestamp']
id missing: 0/500 (0.0%)
id unique: 500/500 (100.0%)
processing_timestamp missing: 438/500 (87.6%)


**Findings**

The dataset contains a stable record identifier (`id`), which supports locating a specific application record in response to a deletion request. However, `processing_timestamp` is missing for most records, limiting the ability to audit when records were processed and whether retention and deletion actions occurred on time. No repository artifacts demonstrate a deletion workflow across derived datasets, logs, and model outputs. This creates a governance gap for Art. 17 because operational deletion capability cannot be evidenced from the dataset and repository alone.

### 8.6 Art. 22 - Automated Decision-Making

Art. 22 applies to decisions based solely on automated processing that produce legal or similarly significant effects. Credit approval and credit pricing fall within this category. Governance expectations include providing meaningful information about the logic involved, enabling contestation, and supporting human review where applicable. This subsection uses recorded rejection reasons as a dataset-level proxy for decision transparency.

In [16]:
# rejection reason distribution and opacity check
n_records = len(df)

required_cols = ["loan_approved", "rejection_reason"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    print(f"missing columns for art. 22 check: {missing_required}")
else:
    rej = df[df["loan_approved"] == 0].copy()
    n_rej = len(rej)

    print(f"rejections: {n_rej}/{n_records} ({n_rej/n_records:.1%})")

    reason_counts = rej["rejection_reason"].value_counts(dropna=False)
    display(reason_counts.to_frame("count"))

    opaque = int((rej["rejection_reason"] == "algorithm_risk_score").sum())
    informative = n_rej - opaque

    print(f"opaque reason 'algorithm_risk_score': {opaque}/{n_rej} ({opaque/n_rej:.1%})")
    print(f"more specific reasons: {informative}/{n_rej} ({informative/n_rej:.1%})")

rejections: 208/500 (41.6%)


,count
rejection_reason,
algorithm_risk_score,169
insufficient_credit_history,23
high_dti_ratio,12
low_income,4


opaque reason 'algorithm_risk_score': 169/208 (81.2%)
more specific reasons: 39/208 (18.8%)


**Findings**

All rejected applications record a `rejection_reason`, but most reasons are not meaningful. Out of 210 rejections, 170 (81.0%) use `algorithm_risk_score`, which does not provide an actionable explanation to a data subject. Only 40 rejections (19.0%) provide more specific reasons. This indicates a transparency gap relevant to Art. 22 safeguards, because rejection messaging is unlikely to support contestation or meaningful human review without additional documentation, clearer reason codes, and defined escalation procedures.

### 8.7 Art. 25 — Data Protection by Design and by Default

Art. 25 requires the controller to implement technical and organisational measures so that only personal data necessary for the purpose is processed by default. In a credit decisioning context, this typically means separating direct identifiers from analytical datasets, restricting access by role, and using pseudonymised identifiers for routine analytics and model development.


| Principle | Art. 25 Requirement | Current State | Gap |
|---|---|---|---|
| Pseudonymisation at ingestion | Art. 25(1) — technical measures to implement Art. 5 | NOT IMPLEMENTED — direct identifiers stored in plaintext | **CRITICAL** |
| Data minimisation by default | Art. 25(2) — only necessary data processed by default | NOT IMPLEMENTED — `gambling`, `adult_entertainment`, `alcohol` collected without credit relevance | **CRITICAL** |
| Purpose limitation at schema level | Art. 25(1) — measures to ensure Art. 5(1)(b) | NOT EVIDENCED — no schema-level purpose tagging or access controls documented | **HIGH** |
| k-anonymity for analytical exports | Art. 25(1) — state-of-the-art techniques (Recital 78) | NOT IMPLEMENTED — dataset k-anonymity = 1 | **CRITICAL** |
| Retention limits enforced technically | Art. 25(2) — data not kept longer than necessary by default | NOT EVIDENCED — no automated deletion or retention flag in dataset | **HIGH** |

**Verdict:** Art. 25 compliance is not evidenced across any of the five assessed dimensions.


### 8.8 Art. 5(1)(e) - Proposed Data Retention Schedule

Art. 5(1)(e) requires that personal data is kept no longer than necessary for the purposes of processing. The dataset does not include fields that encode a retention policy, deletion status, or links to a retention schedule. This subsection proposes a retention schedule aligned to credit underwriting, complaints handling, and auditability needs.

| Data Category | Retention Period | Justification | Legal Basis | Owner |
|---|---|---|---|---|
| Direct identifiers (`full_name`, `email`, `ssn`, `ip_address`) | 90 days post-decision | Minimum necessary for identity verification; pseudonymise after 90 days | GDPR Art. 5(1)(e); Art. 25 | DPO |
| Approved application — financial data and decision | 7 years | AML/KYC obligations under 4AMLD Art. 40; credit dispute resolution window | 4AMLD Art. 40; GDPR Art. 6(1)(c) | Compliance Officer |
| Rejected application — financial data and decision | 2 years | Dispute resolution and fairness monitoring | GDPR Art. 6(1)(f); Art. 5(1)(e) | Compliance Officer |
| Sensitive spending categories | Delete immediately | No documented lawful basis; Art. 9(2) exception not established | GDPR Art. 5(1)(c); Art. 9(1) | DPO |
| Audit logs and decision timestamps | 5 years | EU AI Act Art. 12 audit trail; AML record-keeping | EU AI Act Art. 12; GDPR Art. 5(2) | CTO |
| Pseudonymised analytical dataset | Model lifecycle + 2 years | Required for retraining, fairness monitoring, and post-deployment audit | GDPR Recital 26; EU AI Act Art. 10 | Data Science Team |

No dataset fields indicate retention logic, deletion status, or linkage to a retention schedule. This limits auditability of storage limitation compliance and increases the risk of retaining direct identifiers longer than necessary. A documented retention policy and an operational deletion workflow are required to evidence compliance with Art. 5(1)(e).

### 8.9 GDPR Compliance - Consolidated Risk Assessment

| Finding | Evidence | Regulatory reference | Severity | Governance impact | Recommended control |
|---|---|---|---|---|---|
| Conditional sensitive behavioral spending fields are collected without necessity evidence | alcohol 11 (2.2%), gambling 7 (1.4%), adult entertainment 5 (1.0%) | Art. 5(1)(c) (data minimisation), Art. 5(1)(b) (purpose limitation) | High | Disproportionate privacy risk relative to likely underwriting value if necessity is not documented | Document necessity and purpose, restrict access, and remove from routine analytics if not required |
| Lawful basis and notice traceability are not evidenced in repository artifacts | No fields indicating lawful basis, consent status, or privacy notice version are present | Art. 6, Art. 13, Art. 5(2) | High | Controller cannot demonstrate compliance from dataset and repository artifacts alone | Maintain lawful basis records and privacy notice versioning, and link processing to those records in governance documentation |
| Rejection reasons are recorded but are largely not meaningful | 210 rejections (42.0%); 170 (81.0% of rejections) use `algorithm_risk_score` | Art. 22 safeguards, transparency expectations | Critical | Decision transparency is unlikely to support contestation or meaningful human review | Implement specific reason codes, define escalation and human review process, and log decision rationale consistently |
| Privacy by design and by default is not evidenced at the dataset layer | Direct identifiers present at near-complete coverage; separation of identity data from analytical data is not evidenced | Art. 25, Art. 5(1)(f) | Critical | Broad access or sharing would expose identifiable records without additional controls | Separate identity store from analytical dataset, apply pseudonymisation by default, enforce least-privilege access and access logging |
| Storage limitation and deletion capability are not evidenced | No retention flags, deletion status, or retention policy artifacts are present | Art. 5(1)(e), Art. 5(2) | High | Risk of retaining identifiers longer than necessary and inability to audit compliance | Adopt a retention schedule, implement automated deletion, and maintain deletion audit logs |

## 9. EU AI Act Risk Classification

The EU AI Act (Reg. 2024/1689) establishes a risk-based framework. Credit scoring systems are explicitly listed as high-risk under Annex III §5(b). This section determines NovaCred's risk tier and assesses compliance with the resulting obligations under Chapter III, Section 2 (Art. 9–15).


### 9.1 Risk Tier Classification

**Classification: HIGH-RISK**
**Legal basis:** EU AI Act Annex III, Point 5(b)

> *"AI systems intended to be used to evaluate the creditworthiness of natural persons or establish their credit score, with the exception of AI systems used for the purpose of detecting financial fraud."*

| Dimension | Detail |
|---|---|
| **Provider** | NovaCred (fintech startup) |
| **System** | ML-driven credit decisioning model |
| **Function** | Automated loan approval/rejection and interest rate assignment |
| **Deployment** | Consumer credit applications in the EU |

**Supporting indicators:**

1. Automated approval/rejection has legally significant effects on applicants (access to credit, financial inclusion)
2. System processes protected attributes (gender, age) directly or via proxy variables (`zip_code`), raising Art. 10(5) concerns
3. Bias analysis confirms Disparate Impact ratio = 0.77 (below the 0.80 four-fifths threshold), evidencing discriminatory outcomes by gender

### 9.2 High-Risk Obligations Assessment (Art. 9–15)

| Article | Obligation | Status | Evidence Gap |
|---|---|---|---|
| Art. 9 | Risk Management System | **NOT EVIDENCED** | No documented risk management process found in repository |
| Art. 10 | Data Governance | **PARTIAL** | Bias was detected upstream and protected/proxy attributes were removed in the bias-remediated dataset, but monitoring, governance procedures, and documentation are not evidenced |
| Art. 11 | Technical Documentation | **NOT EVIDENCED** | No model card, data sheet, or technical specification found |
| Art. 12 | Record-Keeping / Audit Logs | **NOT EVIDENCED** | `processing_timestamp` is 87.6% missing — no reliable audit trail |
| Art. 13 | Transparency | **NOT EVIDENCED** | No transparency documentation provided to credit officers |
| Art. 14 | Human Oversight | NOT EVIDENCED | No human review workflow is documented; rejection reasons are largely non-meaningful, limiting contestation and review |
| Art. 15 | Accuracy, Robustness & Cybersecurity | **NOT EVIDENCED** | No model performance metrics, accuracy benchmarks, or adversarial testing documented |


**EU AI Act — Risk Assessment**

| Finding | Evidence | Regulatory Reference | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|---|
| System is high-risk with 6 of 7 obligations unmet | Annex III §5(b) confirmed; Art. 9–15 assessment shows 1 partial, 6 not evidenced | EU AI Act Art. 9–15; Annex III | **Critical** | Non-compliance of a high-risk credit scoring system creates material enforcement risk as the AI Act becomes applicable from 2 August 2026, subject to phased timelines and exceptions | Designate AI system owner; initiate conformity assessment; appoint notified body |
| No risk management system (Art. 9) | No documented risk management process in repository | EU AI Act Art. 9 | **Critical** | Lifecycle risk not managed; errors and bias go undetected | Implement ISO 31000-aligned AI risk management; integrate with model monitoring |
| No audit logs / decision trail (Art. 12) | processing_timestamp missing for majority of records | EU AI Act Art. 12 | **High** | Post-hoc auditing impossible; regulator cannot investigate complaints | Enforce non-nullable timestamp at ingestion; retain logs for 5 years |
| No technical documentation (Art. 11) | No model card, data sheet, or system specification found | EU AI Act Art. 11 | **High** | Cannot demonstrate conformity to market surveillance authority | Produce model card covering training data, performance metrics, bias mitigation |
| Bias confirmed in training outcomes (Art. 10) | DI = 0.77 (below 0.80 threshold); DPD = 0.15; three intersectional violations | EU AI Act Art. 10(2)(f) | **Critical** | Biased system deployed at scale; discriminatory outcomes for protected groups | Suspend deployment; remediate bias; re-evaluate DI before re-deployment |

## 10. Governance Controls & Recommendations

This section proposes concrete governance controls mapped to the GDPR and EU AI Act gaps identified in earlier sections. Controls are prioritised by urgency using three tiers. Critical controls address immediate high-impact gaps that materially increase regulatory exposure. High priority controls reduce significant governance risk and improve auditability. Medium priority controls strengthen operational resilience and long-term compliance.

### 10.1 Critical Priority

| Area | Issue | Recommendation | Regulatory reference |
|---|---|---|---|
| Data minimisation | Conditional sensitive spending fields are collected without necessity evidence | Restrict these fields from routine analytics and modelling until necessity is documented. Remove from the feature set if they do not provide material underwriting value | GDPR Art. 5(1)(c); Art. 5(1)(b) |
| Decision transparency | Most recorded rejection reasons are not meaningful (`algorithm_risk_score`) | Implement a controlled reason-code taxonomy and require a specific, human-interpretable rejection reason for each rejected application. Provide an applicant-facing explanation and a contestation route | GDPR Art. 22 safeguards; EU AI Act Art. 13 |
| Privacy by default | Direct identifiers are present in the analytical dataset at near-complete coverage | Separate identity data from analytical data. Default to a pseudonymised analytical dataset and tightly restrict access to raw identifiers using least privilege and access logging | GDPR Art. 25; Art. 5(1)(f) |

### 10.2 High Priority

| Area | Issue | Recommendation | Regulatory reference |
|---|---|---|---|
| Pseudonymisation | Direct identifiers are stored alongside decision outputs in the same dataset | Apply pseudonymisation by default at ingestion and store raw identifiers in a separate access-restricted identity store. Store pseudonymisation keys in a dedicated key management service | GDPR Art. 25; Art. 5(1)(f); Recital 26 |
| Audit trail | `processing_timestamp` is missing for 87.6% of records, limiting traceability | Enforce non-nullable event timestamps in the decision pipeline and retain decision logs for a defined period aligned to governance requirements | EU AI Act Art. 12; GDPR Art. 5(2) |
| Human oversight | No evidence of a human review workflow for automated decisions | Define a human review process for contested decisions and edge cases. Log review actions, overrides, and reviewer identity (pseudonymised) in an audit trail | EU AI Act Art. 14; GDPR Art. 22 safeguards |
| Fairness governance | Upstream bias audit identified disparate impact by gender before remediation | Implement ongoing fairness monitoring and governance controls to prevent regression. Track fairness metrics over time and define escalation thresholds | EU AI Act Art. 10; Directive 2004/113/EC |

### 10.3 Medium Priority

| Area | Issue | Recommendation | Regulatory reference |
|---|---|---|---|
| Erasure workflow | Repository artifacts do not evidence a deletion workflow across derived data and logs | Implement a documented erasure workflow that propagates deletion to derived datasets and audit logs where applicable. Maintain deletion audit records | GDPR Art. 17 |
| Data retention policy | No retention schedule is evidenced in dataset or repository artifacts | Adopt a retention schedule and automate deletion. Treat direct identifiers separately and delete or tokenise them after the operational decision window | GDPR Art. 5(1)(e); GDPR Art. 5(2) |
| Model documentation | Technical documentation is not evidenced in repository artifacts | Produce technical documentation covering intended use, training data, performance, limitations, and governance controls | EU AI Act Art. 11; Art. 13 |

In [17]:
# GDPR Art. 5(2) — accountability: controller must be able to demonstrate compliance.
# EU AI Act Art. 12 — high-risk AI systems must automatically generate logs enabling post-hoc auditing. 
# This cell defines the minimum-viable audit event schema and demonstrates a compliant vs. non-compliant record against the current dataset.

print("=" * 70)
print("EU AI Act Art. 12 / GDPR Art. 5(2) - Audit Trail Schema")
print("=" * 70)

def build_audit_event(
    applicant_id: str,
    model_version: str,
    decision: str,
    decision_score: float,
    rejection_reason: str | None,
    human_reviewed: bool,
    reviewer_id: str | None,
    override_applied: bool,
    fairness_flag: bool,
) -> dict:
    import uuid

    return {
        "event_id": str(uuid.uuid4()),
        "event_timestamp": pd.Timestamp.utcnow().isoformat() + "Z",
        "applicant_id_hash": pseudonymise_sha256(applicant_id),
        "model_version": model_version,
        "decision": decision,
        "decision_score": round(decision_score, 4),
        "rejection_reason": rejection_reason,
        "human_reviewed": human_reviewed,
        "reviewer_id_hash": pseudonymise_sha256(reviewer_id) if reviewer_id else None,
        "override_applied": override_applied,
        "fairness_flag": fairness_flag,
    }

compliant_event = build_audit_event(
    applicant_id="APP-00042",
    model_version="2.1.3",
    decision="REJECTED",
    decision_score=0.23,
    rejection_reason="high_dti_ratio",
    human_reviewed=False,
    reviewer_id=None,
    override_applied=False,
    fairness_flag=False,
)

non_compliant_event = build_audit_event(
    applicant_id="APP-00099",
    model_version="2.1.3",
    decision="REJECTED",
    decision_score=0.21,
    rejection_reason="algorithm_risk_score",
    human_reviewed=False,
    reviewer_id=None,
    override_applied=False,
    fairness_flag=False,
)

print("\nCompliant audit event:")
display(pd.DataFrame(compliant_event.items(), columns=["field", "value"]))

print("\nNon-compliant audit event example (opaque rejection_reason):")
display(pd.DataFrame(non_compliant_event.items(), columns=["field", "value"]))

n_records = len(df)
missing_timestamp = int(df["processing_timestamp"].isna().sum()) if "processing_timestamp" in df.columns else 0

rej = df[df["loan_approved"] == 0] if "loan_approved" in df.columns else pd.DataFrame()
n_rej = int(len(rej))
opaque = int((rej["rejection_reason"] == "algorithm_risk_score").sum()) if n_rej > 0 else 0

print(f"\nmissing processing_timestamp: {missing_timestamp}/{n_records} ({missing_timestamp/n_records:.1%})")
print(f"rejections: {n_rej}/{n_records} ({n_rej/n_records:.1%})")
print(f"opaque rejection_reason 'algorithm_risk_score': {opaque}/{n_rej} ({opaque/n_rej:.1%})" if n_rej > 0 else "opaque rejection_reason: n/a")

EU AI Act Art. 12 / GDPR Art. 5(2) - Audit Trail Schema

Compliant audit event:


/var/folders/c_/rvj39p2n57b4_31w08d73h3c0000gn/T/ipykernel_99193/4150584651.py:24: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "event_timestamp": pd.Timestamp.utcnow().isoformat() + "Z",


,field,value
0,event_id,35cffcb8-7909-4284-afd7-d890badaacde
1,event_timestamp,2026-03-05T21:24:41.132627+00:00Z
2,applicant_id_hash,beab5b4293f636753c23c0045b9484aad5b3f62624fde3...
3,model_version,2.1.3
4,decision,REJECTED
5,decision_score,0.23
6,rejection_reason,high_dti_ratio
7,human_reviewed,False
8,reviewer_id_hash,None
9,override_applied,False



Non-compliant audit event example (opaque rejection_reason):


,field,value
0,event_id,760799b6-df32-4242-8037-d62874976ee3
1,event_timestamp,2026-03-05T21:24:41.133234+00:00Z
2,applicant_id_hash,ea689a7cbc91381a976b12cd90b0fbc1c509463fe92c69...
3,model_version,2.1.3
4,decision,REJECTED
5,decision_score,0.21
6,rejection_reason,algorithm_risk_score
7,human_reviewed,False
8,reviewer_id_hash,None
9,override_applied,False



missing processing_timestamp: 438/500 (87.6%)
rejections: 208/500 (41.6%)
opaque rejection_reason 'algorithm_risk_score': 169/208 (81.2%)


**Audit Trail Risk Assessment**

| Finding | Evidence | Regulatory reference | Severity | Governance impact | Recommended control |
|---|---|---|---|---|---|
| No reliable audit trail exists | `processing_timestamp` missing for 87.6% of records | EU AI Act Art. 12; GDPR Art. 5(2) | Critical | Post-hoc auditing and complaint investigation cannot be reliably performed | Enforce non-nullable event timestamps and store append-only decision logs |
| Rejection reasons are largely not meaningful | `algorithm_risk_score` for 170 of 210 rejections (81.0%) | GDPR Art. 22 safeguards; EU AI Act Art. 13 | Critical | Decision transparency is insufficient for contestation and human review | Implement reason-code taxonomy and require human-interpretable reason codes |
| Human review actions are not recorded | No fields indicating review, overrides, or reviewer are present in the dataset | EU AI Act Art. 14 | High | Human oversight cannot be demonstrated to auditors or regulators | Log human review, overrides, reviewer identity (pseudonymised), and escalation triggers |
| Retention and integrity of logs is not evidenced | No log retention policy artifacts are present | EU AI Act Art. 12; GDPR Art. 5(2) | High | Logs may be deleted or altered without auditability | Implement append-only storage and retention controls aligned to governance policy |

### 10.4 DPIA Requirement (GDPR Art. 35)

A Data Protection Impact Assessment (DPIA) is required for NovaCred under GDPR Art. 35(3)(a) because the system performs systematic and extensive automated processing that produces significant effects on applicants. The high-risk classification under the EU AI Act further reinforces the expectation that a DPIA is required before operational deployment.

**GDPR Art. 35(7) DPIA elements**

| # | Required element | Reference | NovaCred status |
|---|---|---|---|
| 1 | Systematic description of processing operations and purposes | Art. 35(7)(a) | Not evidenced |
| 2 | Assessment of necessity and proportionality relative to those purposes | Art. 35(7)(b) | Not evidenced |
| 3 | Assessment of risks to rights and freedoms of data subjects | Art. 35(7)(c) | Not evidenced |
| 4 | Measures to address the risks, including safeguards and security measures | Art. 35(7)(d) | Partial, this audit proposes controls but implementation is not evidenced |
| 5 | Controllers, joint controllers, and processors involved | Art. 35(7)(d) | Not evidenced |
| 6 | Consultation with data subjects or their representatives where appropriate | Art. 35(9) | Not evidenced |
| 7 | DPO involvement and documented opinion | Art. 35(2) | Not evidenced |
| 8 | Prior consultation with the supervisory authority if residual risk remains high | Art. 36(1) | Conditional, required if residual risk remains high after mitigation |
| 9 | Scheduled review when the risk profile changes | Art. 35(11) | Not evidenced |

**Action:** No further model deployment should occur until elements 1–4 are complete. If residual risk cannot be reduced to an acceptable level after implementing the controls in Section 10.1–10.3, prior consultation with the competent DPA under Art. 36(1) is required before the system re-enters production.

## 11. Data Remediation

This section produces a privacy-reduced analytical dataset by removing remaining direct identifiers and conditional sensitive behavioral fields from the bias-remediated dataset. Protected attributes and key proxy variables were removed upstream by `02-bias-analysis.ipynb`. This step focuses on reducing re-identification exposure and supporting data minimisation for downstream analysis and reporting.

**Pipeline handoff**

| Step | Notebook | Columns removed | Output file |
|---|---|---|---|
| 1 Data quality | `01-data-quality.ipynb` | Deduplication and normalisation | `cleaned_credit_applications.parquet` |
| 2 Bias remediation | `02-bias-analysis.ipynb` | Protected attributes and key proxies (`gender`, `date_of_birth`, `age`, `zip_code`) | `bias_remediated_credit_applications.parquet` |
| 3 Privacy remediation | `03-privacy-demo.ipynb` (this step) | Direct identifiers and conditional sensitive behavioral fields | `remediated_credit_applications.parquet` |

**Columns removed in this step**

| Category | Columns | Rationale |
|---|---|---|
| Direct identifiers | `full_name`, `email`, `ssn`, `ip_address` | Reduce re-identification exposure and support privacy by default and confidentiality |
| Conditional sensitive behavioral fields | `spending_adult_entertainment`, `spending_gambling`, `spending_alcohol` | Data minimisation due to sensitive inference risk and unclear necessity for underwriting |

In [18]:
# protected attributes and key proxies are already absent
# this step removes direct identifiers and conditional sensitive behavioral fields

PII_COLS = ["full_name", "email", "ssn", "ip_address"]

SENSITIVE_BEHAVIORAL_COLS = [
    "spending_adult_entertainment",
    "spending_gambling",
    "spending_alcohol",
]

COLS_TO_DROP = PII_COLS + SENSITIVE_BEHAVIORAL_COLS
existing_to_drop = [c for c in COLS_TO_DROP if c in df.columns]

df_remediated = df.drop(columns=existing_to_drop).copy()

print(f"bias-remediated input:   {df.shape[0]} rows x {df.shape[1]} columns")
print(f"columns removed:         {len(existing_to_drop)}")
print(f"remediated output:       {df_remediated.shape[0]} rows x {df_remediated.shape[1]} columns")
print(f"dropped columns:         {sorted(existing_to_drop)}")

bias-remediated input:   500 rows x 38 columns
columns removed:         7
remediated output:       500 rows x 31 columns
dropped columns:         ['email', 'full_name', 'ip_address', 'spending_adult_entertainment', 'spending_alcohol', 'spending_gambling', 'ssn']


In [19]:
# verification check to confirm removed columns are absent
remaining = [c for c in COLS_TO_DROP if c in df_remediated.columns]
if remaining:
    print(f"warning: columns still present: {remaining}")
else:
    print("verification passed: no direct identifiers or conditional sensitive behavioral fields remain")

verification passed: no direct identifiers or conditional sensitive behavioral fields remain


In [20]:
# Export the remediated dataset — authoritative output of the Governance Officer audit
remediated_path = "../data/processed/remediated_credit_applications.parquet"
os.makedirs(os.path.dirname(remediated_path), exist_ok=True)
df_remediated.to_parquet(remediated_path, index=False)

print(f"Remediated dataset saved to: {remediated_path}")
print(f"Shape: {df_remediated.shape[0]} rows × {df_remediated.shape[1]} columns")

Remediated dataset saved to: ../data/processed/remediated_credit_applications.parquet
Shape: 500 rows × 31 columns


**Findings**

A privacy-reduced analytical dataset was produced by removing four direct identifiers and three conditional sensitive behavioral fields. This reduces direct re-identification exposure and supports data minimisation for downstream analysis. Remaining attributes still constitute personal data in a credit decisioning context and require appropriate access control, retention enforcement, and audit logging.

## 12. Consolidated Risk Assessment

This section consolidates the key privacy and governance risks identified in this notebook. The assessment reflects what can be evidenced from the bias-remediated dataset and repository artifacts. Items related to the raw dataset are labeled as upstream exposure.

| Area | Finding | Evidence | Regulatory mapping | Severity |
|---|---|---|---|---|
| Re-identification | Direct identifiers present at near-complete coverage | `full_name`, `email`, `ssn`, `ip_address` are present and each is populated for the vast majority of records | GDPR Art. 4(1); Art. 25; Art. 5(1)(f) | Critical |
| Re-identification | Upstream quasi-identifiers were present in raw data | `date_of_birth`, `zip_code`, `gender` were present upstream and removed during bias remediation | GDPR Art. 4(1) | High |
| Data minimisation | Conditional sensitive behavioral fields collected | alcohol 11 (2.2%), gambling 7 (1.4%), adult entertainment 5 (1.0%) | GDPR Art. 5(1)(c); Art. 5(1)(b) | High |
| Special-category inference | Health-related inference via `loan_purpose` | `loan_purpose` populated 50 (10.0%); `medical` 8 (1.6%) | GDPR Art. 9 conditional; Art. 5(1)(c) | High |
| Decision transparency | Rejection reasons are largely not meaningful | 210 rejections (42.0%); `algorithm_risk_score` 170 of 210 (81.0%) | GDPR Art. 22 safeguards; EU AI Act Art. 13 | Critical |
| Accountability | Lawful basis and notice traceability not evidenced | No fields indicating lawful basis, consent status, or privacy notice versioning are present | GDPR Art. 6; Art. 13; Art. 5(2) | High |
| Auditability | Weak dataset-level traceability | `processing_timestamp` missing 438 (87.6%) | GDPR Art. 5(2); EU AI Act Art. 12 | High |
| Storage limitation | Retention policy not evidenced | No retention flags, deletion status, or retention schedule artifacts are present | GDPR Art. 5(1)(e); Art. 5(2) | High |
| Human oversight | Human review workflow not evidenced | No fields or repository artifacts indicate human review, overrides, or escalation | GDPR Art. 22 safeguards; EU AI Act Art. 14 | High |
| EU AI Act classification | High-risk AI system | Creditworthiness assessment system classified as high-risk (Annex III, point 5(b)) | EU AI Act Annex III, 5(b) | Critical |

**Overall consolidated risk: Critical.**

## 13. Audit Limitations

This audit is constrained by what can be evidenced from the dataset and repository artifacts alone. The following dimensions cannot be assessed from these materials and must be addressed through additional due diligence.

| Dimension | Why it cannot be assessed | Recommended action |
|---|---|---|
| Consent mechanisms | No consent records or data subject agreement logs are present | Obtain and review the consent framework and privacy notice process, including Art. 13 and Art. 14 disclosures |
| Access controls | No access logs or role-based access control evidence is available | Request the access control matrix and audit logging configuration from the engineering team |
| Data processing agreements | Third-party processor arrangements are not visible | Audit data processing agreements with cloud providers and external vendors |
| Model internals | Feature importance, thresholds, and training provenance are not available | Require a model card and conduct a technical review with access to model artifacts |
| End-to-end pipeline | Data ingestion, model serving, and output logging are not documented | Commission a system architecture and data flow review |
| Human override history | No evidence of human review or override decisions is recorded | Instrument the decision system to log human interventions and escalation outcomes |
| Prior regulatory correspondence | No regulatory filings or supervisory authority communications are available | Request records from legal and compliance owners |

These limitations should be documented and addressed within the DPIA process under GDPR Art. 35, which is expected for automated credit decisioning that produces significant effects on applicants.

## 14. Conclusions

Overall privacy risk classification: High

This audit identified material privacy and governance gaps in the NovaCred credit decisioning pipeline. The primary drivers are the presence of direct identifiers in the analytical dataset, limited dataset-level accountability signals, and weak decision transparency for rejections. In addition, the system qualifies as high-risk under the EU AI Act because it performs automated creditworthiness assessment for natural persons, which elevates documentation, logging, transparency, and human oversight expectations.

**Prioritised action plan**

Immediate (0 to 30 days)

| # | Action | Owner | Regulatory driver |
|---|---|---|---|
| 1 | Restrict conditional sensitive behavioral spending fields from routine analytics and modelling until necessity is documented. Remove from the feature set if not required | DPO and Data Science | GDPR Art. 5(1)(c); Art. 5(1)(b) |
| 2 | Enforce privacy by default at the dataset layer by separating identity data from analytical data and limiting access to raw identifiers | Engineering and DPO | GDPR Art. 25; Art. 5(1)(f) |
| 3 | Replace opaque rejection reasons with a controlled reason-code taxonomy and applicant-facing explanations, with a defined contestation and escalation route | Engineering and Compliance | GDPR Art. 22 safeguards; EU AI Act Art. 13 |
| 4 | Initiate the DPIA process for automated credit decisioning and document necessity, proportionality, risks, and mitigations | DPO | GDPR Art. 35 |

Short-term (1 to 3 months)

| # | Action | Owner | Regulatory driver |
|---|---|---|---|
| 5 | Apply pseudonymisation by default at ingestion and store raw identifiers separately with strict access control. Store keys in a dedicated key management service | Engineering | GDPR Art. 25; Recital 26 |
| 6 | Establish an export control policy for analytical datasets. Prefer aggregation. Where row-level exports are required, enforce k-threshold checks and consider l-diversity for sensitive inference risk | Data Science | GDPR Recital 26; privacy engineering best practice |
| 7 | Implement complete audit logging for decisions and events, including non-nullable timestamps and retention aligned to governance policy | Engineering | EU AI Act Art. 12; GDPR Art. 5(2) |
| 8 | Adopt and operationalise a retention schedule with automated deletion and deletion audit logs | Compliance and Engineering | GDPR Art. 5(1)(e); GDPR Art. 5(2) |
| 9 | Prevent reintroduction of protected attributes and high-risk proxies into modelling features without governance review | Data Science and DPO | EU AI Act Art. 10; Directive 2004/113/EC |

Long-term (3 to 12 months)

| # | Action | Owner | Regulatory driver |
|---|---|---|---|
| 10 | Produce technical documentation, including intended use, training data provenance, performance metrics, and limitations | Data Science | EU AI Act Art. 11; Art. 13 |
| 11 | Implement a documented human oversight process for contestation, overrides, and escalation. Log review actions in an audit trail | Product and Compliance | EU AI Act Art. 14; GDPR Art. 22 safeguards |
| 12 | Prepare for EU AI Act high-risk conformity expectations, including governance roles, monitoring, and internal compliance processes | Legal and DPO | EU AI Act Chapter III, Section 2 |
| 13 | Establish periodic fairness monitoring and escalation thresholds to detect regression over time | Data Science | EU AI Act Art. 9 and Art. 10 |

**Key conclusions**

Direct identifiers remain present in the analytical dataset, which enables re-identification without reliance on quasi-identifier combinations. Conditional sensitive behavioral fields are present and create disproportionate privacy risk unless necessity and access restrictions are documented. Rejection reasons are recorded for all rejected applications, but most reasons are not meaningful, limiting decision transparency and contestation in practice. At the governance level, lawful basis traceability, retention controls, deletion workflows, and human oversight mechanisms are not evidenced in repository artifacts, which creates high compliance risk under GDPR and the EU AI Act.

**Overall privacy risk: High**